Track 2 — Training from Scratch Orchestrator

Executes thin notebook orchestrator calling modular Python scripts under TASK2_slm_from_scratch/scripts/.

Sync Repository: Clones or pulls latest code from remote repository.

In [ ]:
!git clone https://github.com/DevaNandanJS/Benchmarking-LLM-fine-tuning-vs-training-from-scratch-using-the-same-dataset.git llm_task 2>/dev/null || (cd llm_task && git pull)
%cd llm_task

Install Dependencies: Installs required packages and logs environment versions.

In [ ]:
!pip install -q -r requirements.txt
# Freeze the exact versions to TASK2_slm_from_scratch/logs/environment.txt
# (reproducibility lock — commit this file back to the repo)
import os
os.makedirs('TASK2_slm_from_scratch/logs', exist_ok=True)
!pip freeze > TASK2_slm_from_scratch/logs/environment.txt
print('Dependency snapshot written to TASK2_slm_from_scratch/logs/environment.txt')

Hardware Verification: Checks GPU attachment and prints VRAM memory details.

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU detected — connect a Colab GPU runtime first'
props = torch.cuda.get_device_properties(0)
print('GPU:         ', torch.cuda.get_device_name(0))
print('VRAM (GB):   ', round(props.total_memory / 1e9, 2))
print('torch:       ', torch.__version__)
import tokenizers
print('tokenizers:  ', tokenizers.__version__)

Phase 1 — Custom Tokenizer Training

Sweeps Vocab Sizes: Trains Byte-Pair Encoding (BPE) tokenizer candidates across vocab sizes (256, 1024, 4096).

Selects Best Tokenizer: Evaluates fertility metrics to select optimal vocabulary size and saves tokenizer artifacts.

In [ ]:
!python TASK2_slm_from_scratch/scripts/train_tokenizer.py

Inspect Tokenizer Results: Displays vocabulary sweep metrics and selected tokenizer configuration.

In [ ]:
import pandas as pd
df = pd.read_csv('TASK2_slm_from_scratch/eval/vocab_sweep.csv')
print(df.to_string(index=False))
print()
print(open('TASK2_slm_from_scratch/configs/tokenizer_choice.md').read())

Phase 2 — Dataset Construction

Tokenizes Corpus: Processes extracted raw text into contiguous token ID sequences using trained BPE tokenizer.

Splits & Chunks Tensors: Creates 85/15 train and validation tensor splits saved to data/processed/.

In [ ]:
!python TASK2_slm_from_scratch/scripts/build_dataset.py

Inspect Dataset Results: Displays tensor shapes, chunk counts, and dataset split statistics.

In [ ]:
import json as _json
import torch

stats = _json.load(open("data/processed/slm_dataset_stats.json"))
print("=== Track 2 Dataset Stats ===")
for k, v in stats.items():
    sv = str(v)
    print(f"  {k}: {sv[:80]}..." if len(sv) > 80 else f"  {k}: {sv}")

train = torch.load("data/processed/slm_train.pt", weights_only=True)
val   = torch.load("data/processed/slm_val.pt",   weights_only=True)
print("\ntrain input_ids shape:", tuple(train["input_ids"].shape))
print("val   input_ids shape:", tuple(val["input_ids"].shape))
print("chars/window:", stats["chars_per_window"], "raw chars per training example")

print("\n=== Split Strategy (excerpt) ===")
print(open("TASK2_slm_from_scratch/configs/split_strategy.md").read()[:800])


Phase 3 — Model Architecture Implementation

Defines Custom Transformer: Implements decoder-only Small Language Model architecture with causal self-attention.

Validates Architecture: Runs suite of 8 unit tests and outputs model architecture configuration and parameter counts.

In [ ]:
# Cell 3b — Run unit tests + dump architecture artifacts
# Runs all 8 unit tests unconditionally; writes JSON artifacts only on success.
# If any test raises AssertionError, execution stops here and Cell 3c will
# FileNotFoundError — fix the failing test before proceeding.
!python TASK2_slm_from_scratch/scripts/model.py

In [ ]:
# Cell 3c — Inspect architecture config and parameter breakdown
# These files are written by Cell 3b AFTER all unit tests pass.
# FileNotFoundError here means Cell 3b failed — check its output above.
import json as _json

cfg = _json.load(open('TASK2_slm_from_scratch/configs/run_phase3_model.json'))
print('=== Phase 3 Model Config ===')
for k, v in cfg.items():
    print(f'  {k}: {v}')

params = _json.load(open('TASK2_slm_from_scratch/configs/trainable_params.json'))
print('\n=== Parameter Count ===')
for k, v in params.items():
    if isinstance(v, int):
        print(f'  {k}: {v:,}')
    else:
        print(f'  {k}: {v}')

Phase 4 — Training Loop

Executes Architectural & LR Sweeps: Runs 3 training sweeps (small, base, base_highlr) to evaluate scaling dynamics.

Saves Checkpoints & Logs: Records step-level loss and learning rate metrics to logs and saves best validation checkpoints.

Pre-Flight Smoke Test: Runs fast CPU validation pass to verify gradient updates and loss computation.

In [ ]:
# Smoke-test: 4 chunks, 5 steps, CPU fp32.
# Run this locally before pushing to Colab to catch shape/dtype bugs early.
!python TASK2_slm_from_scratch/scripts/train.py --run base --smoke-test

Sweep Run 1 (small): Trains 4-layer model configuration (128 embedding dim, lr=3e-4).

In [ ]:
!python TASK2_slm_from_scratch/scripts/train.py --run small

Sweep Run 2 (base): Trains 6-layer model configuration (6-layer, 192 embedding dim, lr=3e-4).

In [ ]:
!python TASK2_slm_from_scratch/scripts/train.py --run base

Sweep Run 3 (base_highlr): Trains 6-layer model configuration with higher learning rate (lr=6e-4).

In [ ]:
!python TASK2_slm_from_scratch/scripts/train.py --run base_highlr

Inspect Sweep Results: Displays summary comparison table of validation losses across all training sweeps.

In [ ]:
import pandas as pd
df = pd.read_csv('TASK2_slm_from_scratch/eval/sweep_results.csv')
print(df.to_string(index=False))
print(f"\nBest run: {df.loc[df['best_val_loss'].idxmin(), 'run_name']}  "
      f"(best_val_loss={df['best_val_loss'].min():.4f})")

Inspect Checkpoint Configs: Displays detailed metadata and step metrics for top-performing model checkpoints.

In [ ]:
import json as _json, os
for run in ['small', 'base', 'base_highlr']:
    cfg_path = f'TASK2_slm_from_scratch/checkpoints/best_val/{run}/best_val_config.json'
    if os.path.exists(cfg_path):
        cfg = _json.load(open(cfg_path))
        print(f"\n=== {run} best checkpoint ===")
        for k in ['run_name', 'n_layer', 'n_embd', 'learning_rate',
                  'best_val_loss', 'best_val_step', 'total_steps', 'timestamp']:
            print(f"  {k}: {cfg.get(k, 'N/A')}")
    else:
        print(f"\n[{run}] best_val_config.json not found -- run Cell 4c/4d/4e first")

Phase 5 — Quantitative Evaluation

Evaluates Best Checkpoint: Computes validation cross-entropy loss, perplexity, and Bits-Per-Byte (BPB) metrics.

Plots Loss Curves: Generates comparative training and validation loss curves saved to eval/loss_curve.png.

In [ ]:
# Cell 5a -- Smoke-test (CPU, no checkpoint or real data needed)
# Verifies the BPB accumulation logic, plotting, and JSON writing.
!python TASK2_slm_from_scratch/scripts/eval.py --smoke-test

In [ ]:
# Cell 5b -- Run full quantitative evaluation (GPU required)
# Reads sweep_results.csv to auto-detect the best run.
# Pass --run <name> to override: !python ... --run base
!python TASK2_slm_from_scratch/scripts/eval.py

In [ ]:
# Cell 5c -- Inspect evaluation outputs
import json as _json

metrics = _json.load(open('TASK2_slm_from_scratch/eval/final_metrics.json'))
print('=== Track 2 Final Metrics ===')
for k, v in metrics.items():
    sv = str(v)
    print(f'  {k}: ' + (sv[:100] + '...' if len(sv) > 100 else sv) + '')

bpb_gap = metrics['bpb'] - 1.309722
print(f"\nBPB = {metrics['bpb']}  (Track 1 = 1.309722  gap = {bpb_gap:+.6f})")
print('\n=== Loss Curve Interpretation (excerpt) ===')
print(open('TASK2_slm_from_scratch/eval/loss_curve_interpretation.md').read())


Phase 6 — Qualitative Evaluation (Generation)

Generates Sample Completions: Uses identical 8 validation prompts to generate sampling and greedy text completions.

Outputs Sample Artifact: Saves generated text outputs to generations/slm_samples.md for qualitative review.

In [ ]:
# Cell 6a -- Smoke-test (CPU, no checkpoint needed)
# Verifies generate() determinism, annotation logic, markdown writing.
!python TASK2_slm_from_scratch/scripts/generate.py --smoke-test

In [ ]:
# Cell 6b -- Run qualitative generation (GPU recommended)
# Uses the same 8 prompts as Track 1 -- controlled comparison.
# Sampling: T=0.8, top_p=0.9  |  Greedy: argmax (T=0)
!python TASK2_slm_from_scratch/scripts/generate.py

In [ ]:
# Cell 6c -- Preview generated samples
print(open('TASK2_slm_from_scratch/generations/slm_samples.md').read())


Phase 7 — Cross-Track Comparison

Stages Track 2 Artifacts: Copies metric JSONs and loss curve plots to shared_eval with slm_* namespacing.

Fills Comparison Matrix: Populates shared_eval/comparison_notes.md with side-by-side benchmarking results between Track 1 and Track 2.

In [ ]:
# Cell 7a -- Smoke-test
!python TASK2_slm_from_scratch/scripts/compare.py --smoke-test

In [ ]:
# Cell 7b -- Run cross-track comparison (CPU, fast)
# Copies artefacts to shared_eval/ and writes comparison_notes.md.
!python TASK2_slm_from_scratch/scripts/compare.py

In [ ]:
# Cell 7c -- Preview comparison notes
print(open('shared_eval/comparison_notes.md').read())
